In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (

    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.layers import GRU
from tensorflow.keras.optimizers import (
    Adam,
    RMSprop
)

In [2]:
data=pd.read_csv('/kaggle/input/datasets/manishavelamani/dataset/PJME_preprocessd.csv',parse_dates=['Datetime'],index_col='Datetime')

In [3]:
data.columns

Index(['PJME_MW', 'PJME_MW_Scaled', 'Hour', 'Day', 'Week', 'Month',
       'DayOfWeek', 'Weekend', 'Lag_1', 'Lag_24', 'Lag_48', 'Lag_168',
       'RollingMean_24', 'RollingStd_24', 'RollingMean_168'],
      dtype='object')

In [4]:
from sklearn.preprocessing import MinMaxScaler

features = [
    'PJME_MW_Scaled',
    'Hour',
    'Day',
    'Week',
    'Month',
    'DayOfWeek',
    'Weekend',
    'Lag_1',
    'Lag_24',
    'Lag_48',
    'Lag_168',
    'RollingMean_24',
    'RollingStd_24',
    'RollingMean_168'
]

target = 'PJME_MW_Scaled'

# Select input features
multivariate_data = data[features].copy()

# Scale ALL input features
feature_scaler = MinMaxScaler()

multivariate_data_scaled = pd.DataFrame(
    feature_scaler.fit_transform(multivariate_data),
    columns=features,
    index=multivariate_data.index
)

multivariate_data_scaled.head()

,PJME_MW_Scaled,Hour,Day,Week,Month,DayOfWeek,Weekend,Lag_1,Lag_24,Lag_48,Lag_168,RollingMean_24,RollingStd_24,RollingMean_168
Datetime,,,,,,,,,,,,,,
2002-01-08 01:00:00,0.433011,0.043478,0.233333,0.019231,0.0,0.166667,0.0,0.486937,0.353052,0.360420,0.462358,0.535733,0.407201,0.421362
2002-01-08 02:00:00,0.409021,0.086957,0.233333,0.019231,0.0,0.166667,0.0,0.433011,0.325625,0.329371,0.427439,0.539989,0.386780,0.421179
2002-01-08 03:00:00,0.399889,0.130435,0.233333,0.019231,0.0,0.166667,0.0,0.409021,0.315255,0.319960,0.399331,0.544309,0.363683,0.421185
2002-01-08 04:00:00,0.405058,0.173913,0.233333,0.019231,0.0,0.166667,0.0,0.399889,0.316029,0.315750,0.385154,0.548853,0.338064,0.421382
2002-01-08 05:00:00,0.427316,0.217391,0.233333,0.019231,0.0,0.166667,0.0,0.405058,0.336522,0.319496,0.390045,0.553487,0.312793,0.421752


In [5]:
train_size = int(len(multivariate_data_scaled) * 0.70)
val_size = int(len(multivariate_data_scaled) * 0.10)

train = multivariate_data_scaled.iloc[:train_size]
validation = multivariate_data_scaled.iloc[train_size:train_size + val_size]
test = multivariate_data_scaled.iloc[train_size + val_size:]

In [6]:
def create_multivariate_sequences(df, sequence_length, forecast_horizon=24, target_col='PJME_MW_Scaled'):
    X = []
    y = []

    values = df.values
    target_index = df.columns.get_loc(target_col)

    for i in range(len(df) - sequence_length - forecast_horizon + 1):
        X.append(values[i:i + sequence_length])

        y.append(
            values[
                i + sequence_length:
                i + sequence_length + forecast_horizon,
                target_index
            ]
        )

    return np.array(X), np.array(y)

In [7]:
sequence_length = 168

X_train, y_train = create_multivariate_sequences(train, sequence_length)
X_val, y_val = create_multivariate_sequences(validation, sequence_length)
X_test, y_test = create_multivariate_sequences(test, sequence_length)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

print('X_val:', X_val.shape)
print('y_val:', y_val.shape)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (101465, 168, 14)
y_train: (101465, 24)
X_val: (14331, 168, 14)
y_val: (14331, 24)
X_test: (28855, 168, 14)
y_test: (28855, 24)


In [8]:
from sklearn.preprocessing import MinMaxScaler
# Create scaler using the original MW values
scaler = MinMaxScaler()
scaler.fit(data[['PJME_MW']])

MinMaxScaler()

In [9]:
def evaluate_model(model, X_test, y_test, scaler, sequence_length, model_name):
    predictions = model.predict(X_test, verbose=0)
    # Convert scaled values back to original MW values
    pred_original = scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
    y_original = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    mae = mean_absolute_error(y_original.flatten(),pred_original.flatten())
    mse = mean_squared_error(y_original.flatten(),pred_original.flatten())
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_original.flatten(),pred_original.flatten()) * 100
    r2 = r2_score(y_original.flatten(),pred_original.flatten())
    bias = np.mean(pred_original.flatten() - y_original.flatten())
    return {
        'Sequence Length': sequence_length,
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'Bias': bias
    }

In [10]:
gru_phase6 = Sequential([
    GRU(32, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_phase6.compile(
    optimizer='adam',
    loss='mse'
)

gru_phase6.summary()

gru_baseline_history = gru_phase6.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

baseline_gru_result = evaluate_model(
    gru_phase6,
    X_test,
    y_test,
    scaler,
    168,
    'Baseline GRU (Multivariate)'
)

I0000 00:00:1786451354.436399      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 32)             │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,400 (21.09 KB)

 Trainable params: 5,400 (21.09 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0105 - val_loss: 0.0046
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0035 - val_loss: 0.0039
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0031 - val_loss: 0.0038
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0027 - val_loss: 0.0032
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0026 - val_loss: 0.0031
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0025 - val_loss: 0.0032


In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

gru_early = Sequential([
    GRU(32, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_early.compile(
    optimizer='adam',
    loss='mse'
)

gru_early_history = gru_early.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

gru_early_result = evaluate_model(
    gru_early,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + EarlyStopping'
)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0092 - val_loss: 0.0058
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0034 - val_loss: 0.0042
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0028 - val_loss: 0.0036
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0026 - val_loss: 0.0031
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0025 - val_loss: 0.0032


In [14]:
gru_dropout = Sequential([
    GRU(32, input_shape=(sequence_length, X_train.shape[2])),
    Dropout(0.2),
    Dense(24)
])

gru_dropout.compile(
    optimizer='adam',
    loss='mse'
)

gru_dropout_history = gru_dropout.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

gru_dropout_result = evaluate_model(
    gru_dropout,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + Dropout'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0154 - val_loss: 0.0044
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0048 - val_loss: 0.0038
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0043 - val_loss: 0.0037
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0042 - val_loss: 0.0036
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0041 - val_loss: 0.0036
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0040 - val_loss: 0.0033
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0039 - val_loss: 0.0033
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 0.0039 - val_loss: 0.0035
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0038 - val_loss: 0.0034
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 0.0038 - val_loss: 0.0033


In [15]:
gru_more_units = Sequential([
    GRU(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_more_units.compile(
    optimizer='adam',
    loss='mse'
)

gru_more_units_history = gru_more_units.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

gru_more_units_result = evaluate_model(
    gru_more_units,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + More Units'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0070 - val_loss: 0.0039
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0031 - val_loss: 0.0037
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0026 - val_loss: 0.0031
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0025 - val_loss: 0.0030
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0024 - val_loss: 0.0032
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 31s 10ms/step - loss: 0.0023 - val_loss: 0.0029
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0023 - val_loss: 0.0030


In [16]:
gru_batchnorm = Sequential([
    GRU(64, input_shape=(sequence_length, X_train.shape[2])),
    BatchNormalization(),
    Dense(24)
])

gru_batchnorm.compile(
    optimizer='adam',
    loss='mse'
)

gru_batchnorm_history = gru_batchnorm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

gru_batchnorm_result = evaluate_model(
    gru_batchnorm,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + BatchNormalization'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - loss: 0.0205 - val_loss: 0.0064
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0059 - val_loss: 0.0058
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0052 - val_loss: 0.0056
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0049 - val_loss: 0.0045
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0047 - val_loss: 0.0042
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0045 - val_loss: 0.0040
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0044 - val_loss: 0.0038
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0043 - val_loss: 0.0036
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0041 - val_loss: 0.0035
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0040 - val_loss: 0.0034


In [17]:
gru_rmsprop = Sequential([
    GRU(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='mse'
)

gru_rmsprop_history = gru_rmsprop.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

gru_rmsprop_result = evaluate_model(
    gru_rmsprop,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + RMSprop'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0076 - val_loss: 0.0046
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0041 - val_loss: 0.0045
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0033 - val_loss: 0.0040
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0032 - val_loss: 0.0037
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0030 - val_loss: 0.0037
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0028 - val_loss: 0.0036
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.0028 - val_loss: 0.0035


In [19]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

gru_lr = Sequential([
    GRU(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_lr.compile(
    optimizer='adam',
    loss='mse'
)

gru_lr_history = gru_lr.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop, lr_scheduler]
)

gru_lr_result = evaluate_model(
    gru_lr,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + LR Scheduler'
)

Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0065 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0031 - val_loss: 0.0037 - learning_rate: 0.0010
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0029 - val_loss: 0.0035 - learning_rate: 0.0010
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.0028 - val_loss: 0.0032 - learning_rate: 0.0010
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0026 - val_loss: 0.0032 - learning_rate: 0.0010


In [20]:
gru_bs16 = Sequential([
    GRU(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_bs16.compile(
    optimizer='adam',
    loss='mse'
)

gru_bs16_history = gru_bs16.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16
)

gru_bs16_result = evaluate_model(
    gru_bs16,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + Batch Size 16'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0051 - val_loss: 0.0037
Epoch 2/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0030 - val_loss: 0.0034
Epoch 3/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0028 - val_loss: 0.0033
Epoch 4/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 5/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0025 - val_loss: 0.0034
Epoch 6/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 7/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0023 - val_loss: 0.0030
Epoch 8/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0023 - val_loss: 0.0030
Epoch 9/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0022 - val_loss: 0.0031
Epoch 10/10
6342/6342 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0021 - val_loss: 0.0028


In [21]:
gru_bs64 = Sequential([
    GRU(64, input_shape=(sequence_length, X_train.shape[2])),
    Dense(24)
])

gru_bs64.compile(
    optimizer='adam',
    loss='mse'
)

gru_bs64_history = gru_bs64.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

gru_bs64_result = evaluate_model(
    gru_bs64,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + Batch Size 64'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0091 - val_loss: 0.0043
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0034 - val_loss: 0.0040
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0027 - val_loss: 0.0032
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0026 - val_loss: 0.0034
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0026 - val_loss: 0.0032
Epoch 9/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0025 - val_loss: 0.0031
Epoch 10/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0025 - val_loss: 0.0032


In [22]:
gru_two_layers = Sequential([
    GRU(64, return_sequences=True, input_shape=(sequence_length, X_train.shape[2])),
    GRU(32),
    Dense(24)
])

gru_two_layers.compile(
    optimizer='adam',
    loss='mse'
)

gru_two_layers.summary()

gru_two_layers_history = gru_two_layers.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

gru_two_layers_result = evaluate_model(
    gru_two_layers,
    X_test,
    y_test,
    scaler,
    168,
    'GRU + Two Layers'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_12 (GRU)                    │ (None, 168, 64)        │        15,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_13 (GRU)                    │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,560 (99.84 KB)

 Trainable params: 25,560 (99.84 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 52s 16ms/step - loss: 0.0063 - val_loss: 0.0038
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0031 - val_loss: 0.0036
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0027 - val_loss: 0.0033
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0025 - val_loss: 0.0032
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0024 - val_loss: 0.0031
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0023 - val_loss: 0.0030
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0023 - val_loss: 0.0031
Epoch 9/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0022 - val_loss: 0.0031
Epoch 10/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0021 - val_loss: 0.0030


In [23]:
comparison_gru = pd.DataFrame([
    baseline_gru_result,
    gru_early_result,
    gru_dropout_result,
    gru_more_units_result,
    gru_batchnorm_result,
    gru_rmsprop_result,
    gru_lr_result,
    gru_bs16_result,
    gru_bs64_result,
    gru_two_layers_result
])

comparison_gru = comparison_gru.sort_values('RMSE').reset_index(drop=True)

comparison_gru

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,GRU + Two Layers,1311.694007,3.453777e+06,1858.434145,4.179087,0.913491,224.022202
1,168,GRU + Batch Size 16,1307.147249,3.458081e+06,1859.591551,4.116679,0.913383,16.617464
2,168,GRU + More Units,1311.783774,3.460653e+06,1860.282963,4.157413,0.913318,37.583272
3,168,GRU + Dropout,1338.531041,3.506929e+06,1872.679677,4.248561,0.912159,-113.680676
4,168,GRU + EarlyStopping,1347.601405,3.522081e+06,1876.720924,4.276919,0.911780,73.643956
5,168,Baseline GRU (Multivariate),1360.340355,3.593276e+06,1895.593884,4.350889,0.909997,292.581309
6,168,GRU + Batch Size 64,1359.619214,3.639187e+06,1907.665288,4.353796,0.908847,241.856670
7,168,GRU + RMSprop,1388.035985,3.787699e+06,1946.201173,4.447556,0.905127,296.201855
8,168,GRU + BatchNormalization,1441.893365,3.899280e+06,1974.659407,4.642683,0.902332,474.723388
9,168,GRU + LR Scheduler,1536.719474,4.464645e+06,2112.970667,4.829955,0.888171,-526.784862


In [24]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
# ---------------------------------------------------
# Build GRU model for hyperparameter tuning
# ---------------------------------------------------
def build_gru(hp):

    model = Sequential()

    model.add(
        GRU(
            units=hp.Choice(
                'gru_units',
                values=[32, 64, 128]
            ),
            input_shape=(sequence_length, X_train.shape[2])
        )
    )

    model.add(
        Dropout(
            hp.Choice(
                'dropout_rate',
                values=[0.0, 0.2, 0.3]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                'dense_units',
                values=[32, 64, 128]
            ),
            activation='relu'
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        'learning_rate',
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss='mse',
        metrics=['mae']
    )

    return model


# ---------------------------------------------------
# Hyperparameter search
# ---------------------------------------------------
tuner = kt.RandomSearch(
    build_gru,
    objective='val_loss',
    max_trials=3,
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='gru_168_to_24'
)

tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# ---------------------------------------------------
# Get best hyperparameters
# ---------------------------------------------------
best_hp = tuner.get_best_hyperparameters(1)[0]

print('Best GRU Units:', best_hp.get('gru_units'))
print('Best Dropout:', best_hp.get('dropout_rate'))
print('Best Dense Units:', best_hp.get('dense_units'))
print('Best Learning Rate:', best_hp.get('learning_rate'))


# ---------------------------------------------------
# Rebuild and retrain the best model
# ---------------------------------------------------
final_gru = build_gru(best_hp)

final_gru_history = final_gru.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# ---------------------------------------------------
# Evaluate the retrained final model
# ---------------------------------------------------
final_gru_result = evaluate_model(
    final_gru,
    X_test,
    y_test,
    scaler,
    168,
    'Final Tuned GRU'
)

pd.DataFrame([final_gru_result])


# ---------------------------------------------------
# Save the final tuned model
# ---------------------------------------------------
final_gru.save('/kaggle/working/gru168_phase6_final_tuned.keras')

Trial 3 Complete [00h 02m 29s]
val_loss: 0.004349370952695608

Best val_loss So Far: 0.003658113768324256
Total elapsed time: 00h 08m 02s
Best GRU Units: 128
Best Dropout: 0.0
Best Dense Units: 64
Best Learning Rate: 0.0001
Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - loss: 0.0164 - mae: 0.0875 - val_loss: 0.0061 - val_mae: 0.0601
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0045 - mae: 0.0515 - val_loss: 0.0047 - val_mae: 0.0514
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0036 - mae: 0.0445 - val_loss: 0.0040 - val_mae: 0.0460
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0032 - mae: 0.0417 - val_loss: 0.0038 - val_mae: 0.0448
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0030 - mae: 0.0404 - val_loss: 0.0037 - val_mae: 0.0443


In [25]:
comparison_gru = pd.DataFrame([
    baseline_gru_result,
    gru_early_result,
    gru_dropout_result,
    gru_more_units_result,
    gru_batchnorm_result,
    gru_rmsprop_result,
    gru_lr_result,
    gru_bs16_result,
    gru_bs64_result,
    gru_two_layers_result,
    final_gru_result
])

comparison_gru = comparison_gru.sort_values('RMSE').reset_index(drop=True)

comparison_gru

,Sequence Length,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,168,GRU + Two Layers,1311.694007,3.453777e+06,1858.434145,4.179087,0.913491,224.022202
1,168,GRU + Batch Size 16,1307.147249,3.458081e+06,1859.591551,4.116679,0.913383,16.617464
2,168,GRU + More Units,1311.783774,3.460653e+06,1860.282963,4.157413,0.913318,37.583272
3,168,GRU + Dropout,1338.531041,3.506929e+06,1872.679677,4.248561,0.912159,-113.680676
4,168,GRU + EarlyStopping,1347.601405,3.522081e+06,1876.720924,4.276919,0.911780,73.643956
5,168,Baseline GRU (Multivariate),1360.340355,3.593276e+06,1895.593884,4.350889,0.909997,292.581309
6,168,GRU + Batch Size 64,1359.619214,3.639187e+06,1907.665288,4.353796,0.908847,241.856670
7,168,GRU + RMSprop,1388.035985,3.787699e+06,1946.201173,4.447556,0.905127,296.201855
8,168,GRU + BatchNormalization,1441.893365,3.899280e+06,1974.659407,4.642683,0.902332,474.723388
9,168,GRU + LR Scheduler,1536.719474,4.464645e+06,2112.970667,4.829955,0.888171,-526.784862


In [26]:
comparison_gru.to_csv('/kaggle/working/comparison_gru_final.csv')